In [14]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [15]:
load_dotenv()

True

In [16]:
model = ChatGroq(model="llama-3.1-8b-instant")

In [17]:
MAX_TOKENS = 150

In [18]:
def call_node(state: MessagesState):
    messages = trim_messages(
        state['messages'],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print("Current Token Count -->", count_tokens_approximately(messages))

    for message in messages:
        print(getattr(message, "content", str(message)))

    response = model.invoke(messages)
    return {"messages": [response]}

In [19]:
builder = StateGraph(MessagesState)
builder.add_node("call_node",call_node)
builder.add_edge(START,"call_node")

In [20]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer = checkpointer)

In [21]:
config = {"configurable":{"thread_id":"thread-1"}}

result = graph.invoke({"messages":[{"role":"user","content":"Hi, My name is Nitish"}]},config)


Current Token Count --> 10
Hi, My name is Nitish


In [22]:
result = graph.invoke({"messages":[{"role":"user","content":"I am learning Langgraph"}]},config)
result["messages"][-1].content

Current Token Count --> 48
Hi, My name is Nitish
Nice to meet you, Nitish. Is there something I can help you with or would you like to chat?
I am learning Langgraph


'LangGraph is an interesting topic. LangGraph is a graph-based programming model for languages, which can help in representing the structure of a language and analyzing its properties. \n\nWhat specifically would you like to know or discuss about LangGraph? Are you looking for resources, trying to implement it, or exploring its applications?'

In [23]:
result = graph.invoke({"messages":[{"role":"user","content":"Can you explain short term memory"}]},config)
result["messages"][-1].content

Current Token Count --> 142
Nice to meet you, Nitish. Is there something I can help you with or would you like to chat?
I am learning Langgraph
LangGraph is an interesting topic. LangGraph is a graph-based programming model for languages, which can help in representing the structure of a language and analyzing its properties. 

What specifically would you like to know or discuss about LangGraph? Are you looking for resources, trying to implement it, or exploring its applications?
Can you explain short term memory


'Short-term memory (STM) is a cognitive system with a limited capacity to hold information for a short period of time. It\'s a crucial component of human working memory and plays a vital role in various cognitive tasks, such as:\n\n1. **Attention and focus**: STM helps you focus on relevant information and filter out distractions.\n2. **Learning and memory**: STM serves as a temporary holding area for new information, allowing you to process and consolidate it into long-term memory.\n3. **Language processing**: STM is involved in speech comprehension, sentence parsing, and language production.\n\nKey characteristics of short-term memory:\n\n1. **Limited capacity**: STM can hold a limited amount of information, typically 7 ± 2 chunks of information (the "magical number seven, plus or minus two").\n2. **Duration**: Information in STM is retained for a short period, usually 15-30 seconds.\n3. **Volatility**: Information in STM is easily forgotten if not rehearsed or consolidated into long

In [24]:
result = graph.invoke({"messages":[{"role":"user","content":"What is my name?"}]},config)
result["messages"][-1].content

Current Token Count --> 8
What is my name?


"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"

In [25]:
for item in graph.get_state({"configurable":{"thread_id":"thread-1"}}).values['messages']:
    print(item.content)
    print("-"*120)

Hi, My name is Nitish
------------------------------------------------------------------------------------------------------------------------
Nice to meet you, Nitish. Is there something I can help you with or would you like to chat?
------------------------------------------------------------------------------------------------------------------------
I am learning Langgraph
------------------------------------------------------------------------------------------------------------------------
LangGraph is an interesting topic. LangGraph is a graph-based programming model for languages, which can help in representing the structure of a language and analyzing its properties. 

What specifically would you like to know or discuss about LangGraph? Are you looking for resources, trying to implement it, or exploring its applications?
------------------------------------------------------------------------------------------------------------------------
Can you explain short term memory
---